# Local search (hill climbing, simulated annealing, tabu) — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
DIRS=[(1,0),(-1,0),(0,1),(0,-1)]
def neighbors(p, shape=(5,5), walls=set()):
    for dr,dc in DIRS:
        q=(p[0]+dr,p[1]+dc)
        if 0<=q[0]<shape[0] and 0<=q[1]<shape[1] and q not in walls: yield q
def reconstruct(parent, goal):
    path=[]; x=goal
    while x is not None: path.append(x); x=parent.get(x)
    return path[::-1]
def draw_grid(path=(), explored=(), frontier=(), walls=set(), start=(0,0), goal=(4,4), shape=(5,5), title='grid'):
    img=np.zeros(shape)
    for r,c in walls: img[r,c]=-1
    for r,c in explored: img[r,c]=.35
    for r,c in frontier: img[r,c]=.65
    for r,c in path: img[r,c]=1
    plt.figure(figsize=(4,4)); plt.imshow(img,cmap='viridis',vmin=-1,vmax=1); plt.scatter([start[1]],[start[0]],c='w',edgecolor='k'); plt.scatter([goal[1]],[goal[0]],c='r',edgecolor='k'); plt.title(title); plt.xticks(range(shape[1])); plt.yticks(range(shape[0])); plt.grid(color='w',alpha=.3)
print('setup ok')

## Move among complete candidates instead of building a tree
Local search exposes the trade between greedy improvement, random escape, and memory.

In [ ]:
xs=np.arange(11); f=lambda x:-.25*(x-7)**2+10+2*np.exp(-.5*(x-3)**2); x=0; p=[x]
while True:
 ns=[z for z in [x-1,x+1] if 0<=z<=10]; y=max(ns,key=f)
 if f(y)<=f(x): break
 x=y; p.append(x)
plt.figure(figsize=(6,3)); plt.plot(xs,[f(z) for z in xs],'-o'); plt.plot(p,[f(z) for z in p],'r-o'); plt.title('hill climb')
assert x==7 and len(p)==8

In [ ]:
f2=lambda x:8*np.exp(-.5*(x-3)**2)+10*np.exp(-.5*((x-8)/1.2)**2); x=0; p=[x]
while True:
 ns=[z for z in [x-1,x+1] if 0<=z<=10]; y=max(ns,key=f2)
 if f2(y)<=f2(x): break
 x=y; p.append(x)
plt.figure(figsize=(6,3)); plt.plot(xs,[f2(z) for z in xs],'-o'); plt.plot(p,[f2(z) for z in p],'r-o'); plt.title('local trap')
assert x==3 and f2(8)>f2(x)

In [ ]:
f2=lambda x:8*np.exp(-.5*(x-3)**2)+10*np.exp(-.5*((x-8)/1.2)**2); p=[0,1,2,3,4,5,6,7,8]; deltas=[f2(p[i+1])-f2(p[i]) for i in range(len(p)-1)]
plt.figure(figsize=(6,3)); plt.plot(np.arange(11),[f2(z) for z in range(11)],'-o'); plt.plot(p,[f2(z) for z in p],'r-o'); plt.title('annealing accepts a downhill bridge')
assert p[-1]==8 and min(deltas)<0

In [ ]:
score={0:0,1:3,2:2,3:3,4:0}; x=1; tabu=[]; p=[]
for _ in range(6):
 p.append(x); cand=[z for z in [x-1,x+1] if 0<=z<=4 and z not in tabu]
 if not cand: break
 tabu=(tabu+[x])[-2:]; x=max(cand,key=lambda z:score[z])
plt.figure(figsize=(5,3)); plt.plot(range(5),[score[i] for i in range(5)],'-o'); plt.plot(p,[score[i] for i in p],'r-o'); plt.title('tabu memory')
assert p[:4]==[1,2,3,4]

In [ ]:
final=[]; f2=lambda x:8*np.exp(-.5*(x-3)**2)+10*np.exp(-.5*((x-8)/1.2)**2)
for s0 in range(11):
 x=s0
 while True:
  ns=[z for z in [x-1,x+1] if 0<=z<=10]; y=max(ns,key=f2)
  if f2(y)<=f2(x): break
  x=y
 final.append(x)
plt.figure(figsize=(5,3)); plt.scatter(range(11),final); plt.title('random restart basins')
assert 3 in final and 8 in final